# Project 3: SQL Data Analysis
**Goal:** Use SQL queries to extract insights from a dataset.

**Key Requirements:** SELECT queries, WHERE, ORDER BY, GROUP BY, aggregations (COUNT, SUM, AVG)

**Dataset:** E-commerce orders — 1,200 rows, 14 columns (same dataset as Project 2)

**Engine:** SQLite (runs locally with zero installation — built into Python)

## 1. Setup: Load Data into SQLite

In [ ]:
import pandas as pd
import sqlite3

# Load the Excel dataset
df = pd.read_excel('Dataset_for_Data_Analytics.xlsx')

# Create a SQLite database and load the data into a table called 'orders'
conn = sqlite3.connect('orders.db')
df.to_sql('orders', conn, if_exists='replace', index=False)

print('Database created: orders.db')
print(f'Table "orders" loaded with {len(df)} rows.')

## 2. Helper Function
A small helper to run SQL queries and return results as a clean DataFrame.

In [ ]:
def run_query(sql):
    """Run a SQL query against orders.db and return a DataFrame."""
    return pd.read_sql_query(sql, conn)

## 3. Basic SELECT Queries

In [ ]:
# Preview the table structure
sql = "SELECT * FROM orders LIMIT 5;"
run_query(sql)

In [ ]:
# Select specific columns
sql = """
SELECT OrderID, Date, Product, Quantity, TotalPrice
FROM orders
LIMIT 10;
"""
run_query(sql)

## 4. WHERE — Filtering Data

In [ ]:
# Orders above $2,000
sql = """
SELECT OrderID, Product, TotalPrice, OrderStatus
FROM orders
WHERE TotalPrice > 2000;
"""
high_value = run_query(sql)
print(f'Orders above $2,000: {len(high_value)}')
high_value.head(10)

In [ ]:
# Orders paid by Credit Card AND status Cancelled
sql = """
SELECT OrderID, Date, Product, PaymentMethod, OrderStatus, TotalPrice
FROM orders
WHERE PaymentMethod = 'Credit Card' AND OrderStatus = 'Cancelled';
"""
cc_cancelled = run_query(sql)
print(f'Cancelled Credit Card orders: {len(cc_cancelled)}')
cc_cancelled.head(10)

In [ ]:
# Orders using a coupon code (NOT NULL filter)
sql = """
SELECT OrderID, CouponCode, TotalPrice
FROM orders
WHERE CouponCode IS NOT NULL AND CouponCode != ''
LIMIT 10;
"""
run_query(sql)

## 5. ORDER BY — Sorting Data

In [ ]:
# Top 10 highest-value orders
sql = """
SELECT OrderID, Product, TotalPrice, OrderStatus
FROM orders
ORDER BY TotalPrice DESC
LIMIT 10;
"""
run_query(sql)

In [ ]:
# Lowest 10 order values (potential data quality check)
sql = """
SELECT OrderID, Product, TotalPrice, OrderStatus
FROM orders
ORDER BY TotalPrice ASC
LIMIT 10;
"""
run_query(sql)

## 6. GROUP BY + Aggregations (COUNT, SUM, AVG)

In [ ]:
# Total revenue, order count, and average order value by Product
sql = """
SELECT
    Product,
    COUNT(*) AS OrderCount,
    SUM(TotalPrice) AS TotalRevenue,
    ROUND(AVG(TotalPrice), 2) AS AvgOrderValue
FROM orders
GROUP BY Product
ORDER BY TotalRevenue DESC;
"""
run_query(sql)

In [ ]:
# Order count and revenue by OrderStatus
sql = """
SELECT
    OrderStatus,
    COUNT(*) AS OrderCount,
    SUM(TotalPrice) AS TotalRevenue
FROM orders
GROUP BY OrderStatus
ORDER BY OrderCount DESC;
"""
run_query(sql)

In [ ]:
# Average order value by Payment Method
sql = """
SELECT
    PaymentMethod,
    COUNT(*) AS OrderCount,
    ROUND(AVG(TotalPrice), 2) AS AvgOrderValue
FROM orders
GROUP BY PaymentMethod
ORDER BY AvgOrderValue DESC;
"""
run_query(sql)

In [ ]:
# Monthly revenue trend
sql = """
SELECT
    strftime('%Y-%m', Date) AS Month,
    COUNT(*) AS OrderCount,
    SUM(TotalPrice) AS TotalRevenue
FROM orders
GROUP BY Month
ORDER BY Month;
"""
run_query(sql)

## 7. HAVING — Filtering Grouped Data
Going beyond the basics: `HAVING` filters groups *after* aggregation (unlike `WHERE`, which filters rows before grouping).

In [ ]:
# Referral sources that generated more than $150,000 in total revenue
sql = """
SELECT
    ReferralSource,
    COUNT(*) AS OrderCount,
    SUM(TotalPrice) AS TotalRevenue
FROM orders
GROUP BY ReferralSource
HAVING SUM(TotalPrice) > 150000
ORDER BY TotalRevenue DESC;
"""
run_query(sql)

In [ ]:
# Products with an average order value above the overall average ($1,054)
sql = """
SELECT
    Product,
    COUNT(*) AS OrderCount,
    ROUND(AVG(TotalPrice), 2) AS AvgOrderValue
FROM orders
GROUP BY Product
HAVING AVG(TotalPrice) > 1054
ORDER BY AvgOrderValue DESC;
"""
run_query(sql)

## 8. Percentage Contribution by Category
Calculating each product's share of total revenue using a subquery.

In [ ]:
sql = """
SELECT
    Product,
    SUM(TotalPrice) AS TotalRevenue,
    ROUND(SUM(TotalPrice) * 100.0 / (SELECT SUM(TotalPrice) FROM orders), 2) AS PctOfTotalRevenue
FROM orders
GROUP BY Product
ORDER BY PctOfTotalRevenue DESC;
"""
run_query(sql)

In [ ]:
# Percentage of orders by OrderStatus (e.g. what % are cancelled/returned)
sql = """
SELECT
    OrderStatus,
    COUNT(*) AS OrderCount,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM orders), 2) AS PctOfOrders
FROM orders
GROUP BY OrderStatus
ORDER BY PctOfOrders DESC;
"""
run_query(sql)

## 9. Multi-Level GROUP BY
Combining two dimensions — Product and OrderStatus — for a deeper breakdown.

In [ ]:
sql = """
SELECT
    Product,
    OrderStatus,
    COUNT(*) AS OrderCount
FROM orders
GROUP BY Product, OrderStatus
ORDER BY Product, OrderCount DESC;
"""
run_query(sql)

## 10. Top Customers by Spend

In [ ]:
sql = """
SELECT
    CustomerID,
    COUNT(*) AS NumOrders,
    SUM(TotalPrice) AS TotalSpent
FROM orders
GROUP BY CustomerID
ORDER BY TotalSpent DESC
LIMIT 10;
"""
run_query(sql)

## 11. Key Observations & Summary

In [ ]:
# Pull final numbers for the summary
total_orders = run_query("SELECT COUNT(*) AS c FROM orders").iloc[0]['c']
total_revenue = run_query("SELECT SUM(TotalPrice) AS s FROM orders").iloc[0]['s']
avg_order = run_query("SELECT AVG(TotalPrice) AS a FROM orders").iloc[0]['a']
top_product = run_query("""
    SELECT Product, SUM(TotalPrice) AS rev FROM orders
    GROUP BY Product ORDER BY rev DESC LIMIT 1
""").iloc[0]
top_status = run_query("""
    SELECT OrderStatus, COUNT(*) AS c FROM orders
    GROUP BY OrderStatus ORDER BY c DESC LIMIT 1
""").iloc[0]


print(f"""
Total Orders     : {total_orders:,}
Total Revenue    : ${total_revenue:,.2f}
Avg Order Value  : ${avg_order:,.2f}

Top Revenue Product : {top_product['Product']} (${top_product['rev']:,.2f})
Most Common Status  : {top_status['OrderStatus']} ({top_status['c']} orders)

All queries executed successfully against orders.db using SQLite.
""")

In [ ]:
# Close the connection when done
conn.close()
print('Database connection closed.')